# BAROspection - Pre-work Jupyter Notebook
## Adjust Celestial Coordinates for Assigned Stars

This is a Jupyter Notebook. Press ``Shift``+``Enter`` to execute a ``cell``

In [1]:
#!pip install wikipedia

In [2]:
#!pip install --upgrade astroquery

In [3]:
from astropy.coordinates import EarthLocation, AltAz, SkyCoord, SkyOffsetFrame
from astropy.time import Time
import astropy.units as u
from astropy.io import ascii
import numpy as np
import pandas as pd
# importing the module
import wikipedia as wiki

In [4]:
# ra_dec_offset_v5 and output files naming
ra_dec_offset_version = "v6"
data_folder_name = "data_folder"
output_csvfilename = f"adhoc_ra_dec_offset_{ra_dec_offset_version}.csv"
print(f"ra_dec_offset_version is: {ra_dec_offset_version}")
print(f"data_folder_name is: {data_folder_name}")
print(f"output_csvfilename is: {output_csvfilename}")

ra_dec_offset_version is: v6
data_folder_name is: data_folder
output_csvfilename is: adhoc_ra_dec_offset_v6.csv


In [5]:
from astropy.coordinates import EarthLocation, AltAz, SkyCoord, SkyOffsetFrame
from astropy.time import Time
import astropy.units as u

obs_tel = "BARO"
obs_loc = "San Diego"
obs_lat = 32.6 * u.deg  # for san diego
obs_lon = -116.3 * u.deg # for san diego
obs_hgt = 1131 * u.m # for BARO
safe_lim = 10 * u.deg # Account for BARO Telescop stops 
max_mag = 8 # max star magnitudes to consider
min_ra = 12 # min ra limit for BARO
max_ra = 18 # max ra limit for BARO

print(f"Observers Location is: {obs_loc}")
print(f"Observers Telescope is: {obs_tel}")
print(f"Observers Lattitude is: {obs_lat}")
print(f"Observers Longitude is: {obs_lon}")
print(f"Observers Height is: {obs_hgt}")
print(f"Safe Limit for {obs_tel} is: {safe_lim}")
print(f"Max Mag to Query is: {max_mag}")
print(f"Min RA to Query is: {min_ra}")
print(f"Max RA to Query is: {max_ra}")

# Define observer location
location = EarthLocation.from_geodetic(
    lat=obs_lat, lon=obs_lon, height=obs_hgt
)

# Define observation time
time = Time("2025-06-09 21:30:00")

# Define the celestial object's coordinates (e.g., RA and Dec)
sky_coord = SkyCoord(ra=10 * u.deg, dec=20 * u.deg)

# Create an AltAz frame
altaz_frame = AltAz(obstime=time, location=location)

# Transform the object's coordinates to AltAz
altaz_coord = sky_coord.transform_to(altaz_frame)

# Get the altitude and azimuth
altitude = altaz_coord.alt
azimuth = altaz_coord.az

print(f"Altitude: {altitude:.4f}")
print(f"Azimuth: {azimuth:.4f}")


# Define location and time
#location = EarthLocation(lat='32.7', lon='-116.33', height=0*u.m)
#obstime = Time.now()
#obstime = datetime.time(21, 0)

# AltAz frame for the observer
#altaz_frame = AltAz(obstime=obstime, location=location)

# Determine Declination range
min_dec = location.lat - 90*u.deg + safe_lim
max_dec = location.lat + 90*u.deg - safe_lim
print(f"Observable Declination range: {min_dec.to_string(unit=u.deg)} to {max_dec.to_string(unit=u.deg)}")


Observers Location is: San Diego
Observers Telescope is: BARO
Observers Lattitude is: 32.6 deg
Observers Longitude is: -116.3 deg
Observers Height is: 1131.0 m
Safe Limit for BARO is: 10.0 deg
Max Mag to Query is: 8
Min RA to Query is: 12
Max RA to Query is: 18
Altitude: 7.1947 deg
Azimuth: 289.3402 deg
Observable Declination range: -47d24m00s to 112d36m00s


In [6]:
obs_zen = 90 * u.deg - obs_lat
safe_min = obs_lat -90 * u.deg + safe_lim
safe_max = obs_lat +90 * u.deg - safe_lim
print(f'The Zenith at {obs_loc} is: {obs_zen:0.2f} deg')
print(f'Safe Declination limits at {obs_tel} are: {safe_min:0.2f} deg to {safe_max:0.2f} deg')

The Zenith at San Diego is: 57.40 deg deg
Safe Declination limits at BARO are: -47.40 deg deg to 112.60 deg deg


In [7]:
# set target default name
target_default_name = "HD"

In [8]:
def compute_exposure_time(mag: float) -> float:
    """
    Compute exposure time (in seconds) to reach 50,000 flux
    given the apparent magnitude, using the refit model
    (excluding La Superba).
    """
    a = 0.9325
    b = 1.0569
    c = -12.325
    target_flux = 50000

    log_flux = np.log(target_flux)
    log_exp = (log_flux + a * mag + c) / b
    return np.exp(log_exp)*2.5



In [9]:
def compute_adj_coord(original_ra_, original_dec_, offset_arcmin_, camera_rotation_deg_): 
    # --- Step 1A: Compute sky position angle for image "left" ---
    original_coord = SkyCoord(original_ra_, original_dec_)
    #original_coord_icrs = original_coord.transform_to('icrs')
    
    # --- Step 2: Compute sky position angle for image "left" ---
    sky_PA = (270 - camera_rotation_deg_) * u.deg
    
    print(f'\nsky_PA: {sky_PA}')
    
    # --- Step 3: Offset distance converted to tangent plane components ---
    offset_dist = offset_arcmin_ * u.arcmin
    
    print(f'\noffset_dist: {offset_dist}')
    
    dx = offset_dist * np.sin(sky_PA)
    dy = offset_dist * np.cos(sky_PA)
    
    print(f'\ndx: {dx} dy: {dy}')
          

    # --- Step 4: Define the offset frame centered on the original target ---
    offset_frame = SkyOffsetFrame(origin=original_coord)
    
    print(f'\noffset_frame = {offset_frame}')

    # --- Step 5: Create a coordinate in the offset frame and transform back ---
    offset_coord = SkyCoord(lon=dx, lat=dy, frame=offset_frame)
    new_coord_ = offset_coord.transform_to('icrs')
    
    print(f'\noffset_coord = {offset_coord}')
    print(f'\nnew_coord_ = {new_coord_}')
    
    return new_coord_



In [10]:
# ---  CREATE A DATAFRAME
column_names = ["Name1*","Name2*","RA2000*","D2000*","Pmag~","Exp~","Note1","Note2","NExp~","GetRef","Temp"] 
df = pd.DataFrame(columns=column_names)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)  # Or a large integer like 9999
ridx = 0

In [11]:
print(df)

Empty DataFrame
Columns: [Name1*, Name2*, RA2000*, D2000*, Pmag~, Exp~, Note1, Note2, NExp~, GetRef, Temp]
Index: []


In [12]:
# --- Change User Inputs ---
target_name = "zosma"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg):.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg):.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (168.52708927, 20.52371814)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (168.52708927, 20.52371814)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (168.52708927, 20.52371814)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (168.58487306, 20.50195095)>
Using SkyOffsetFrame for Star zosma 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 11.2351, 20.5237
Original RA(Deg)/Dec: 168.5271, 20.5237
New  RA(Hr)/Dec:  11.2390, 20.5020
New  RA(Deg)/Dec:  168.5849, 20.5020
Delta RA/DEC(min): -0.0578,       0.0218


In [13]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [14]:
star_magnitude = 2.14;  target_alt_name = "HD 102647" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A3'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 2.14: 3.98 s


In [15]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                   Name1*     Name2*   RA2000*   D2000* Pmag~  Exp~ Note1  \
0  zosma_HD 102647_Typ_A3  HD 102647  168.5849  20.5020  2.14  3.98    NA   

  Note2 NExp~ GetRef Temp  
0    NA     1      0       


In [16]:
# --- Change User Inputs ---
target_name = "Arcturus"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (213.9153003, 19.18240916)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (213.9153003, 19.18240916)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (213.9153003, 19.18240916)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (213.97259826, 19.16064265)>
Using SkyOffsetFrame for Star Arcturus 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 14.2610, 19.1824
Original RA(Deg)/Dec: 213.9153, 19.1824
New  RA(Hr)/Dec:  14.2648, 19.1606
New  RA(Deg)/Dec:  213.9726, 19.1606
Delta RA/DEC(min): -3.4379,       1.3060


In [17]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [18]:
star_magnitude = -0.05;  target_alt_name = 'HD 124897' # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K1'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag -0.05: 0.58 s


In [19]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                      Name1*     Name2*   RA2000*   D2000*  Pmag~  Exp~ Note1  \
0     zosma_HD 102647_Typ_A3  HD 102647  168.5849  20.5020   2.14  3.98    NA   
1  Arcturus_HD 124897_Typ_K1  HD 124897  213.9726  19.1606  -0.05  0.58    NA   

  Note2 NExp~ GetRef Temp  
0    NA     1      0       
1    NA     1      0       


In [20]:
# --- Change User Inputs ---
target_name = "Neptune"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (0.12937671, -0.63449956)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (0.12937671, -0.63449956)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (0.12937671, -0.63449956)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (0.18350404, -0.6562569)>
Using SkyOffsetFrame for Star Neptune 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 0.0086, -0.6345
Original RA(Deg)/Dec: 0.1294, -0.6345
New  RA(Hr)/Dec:  0.0122, -0.6563
New  RA(Deg)/Dec:  0.1835, -0.6563
Delta RA/DEC(min): -3.2476,       1.3054


In [21]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [22]:
star_magnitude = 7.74;  target_alt_name = target_default_name # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'NA'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 7.74: 556.18 s


In [23]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                      Name1*     Name2*   RA2000*   D2000*  Pmag~    Exp~  \
0     zosma_HD 102647_Typ_A3  HD 102647  168.5849  20.5020   2.14    3.98   
1  Arcturus_HD 124897_Typ_K1  HD 124897  213.9726  19.1606  -0.05    0.58   
2          Neptune_HD_Typ_NA         HD    0.1835  -0.6563   7.74  556.18   

  Note1 Note2 NExp~ GetRef Temp  
0    NA    NA     1      0       
1    NA    NA     1      0       
2    NA    NA     1      0       


In [24]:
# --- Change User Inputs ---
target_name = "Zosma"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (168.52708927, 20.52371814)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (168.52708927, 20.52371814)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (168.52708927, 20.52371814)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (168.58487305, 20.50195095)>
Using SkyOffsetFrame for Star Zosma 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 11.2351, 20.5237
Original RA(Deg)/Dec: 168.5271, 20.5237
New  RA(Hr)/Dec:  11.2390, 20.5020
New  RA(Deg)/Dec:  168.5849, 20.5020
Delta RA/DEC(min): -3.4670,       1.3060


In [25]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [26]:
star_magnitude = 2.56;  target_alt_name = "HD 97603" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A4'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 2.56: 5.76 s


In [27]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                      Name1*     Name2*   RA2000*   D2000*  Pmag~    Exp~  \
0     zosma_HD 102647_Typ_A3  HD 102647  168.5849  20.5020   2.14    3.98   
1  Arcturus_HD 124897_Typ_K1  HD 124897  213.9726  19.1606  -0.05    0.58   
2          Neptune_HD_Typ_NA         HD    0.1835  -0.6563   7.74  556.18   
3      Zosma_HD 97603_Typ_A4   HD 97603  168.5849  20.5020   2.56    5.76   

  Note1 Note2 NExp~ GetRef Temp  
0    NA    NA     1      0       
1    NA    NA     1      0       
2    NA    NA     1      0       
3    NA    NA     1      0       


In [28]:
star_magnitude = 2.56;  target_alt_name = target_default_name # FIX THIS LINE AND BELOW BASED ON SEARCH
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 2.56: 5.76 s


In [29]:
# --- Change User Inputs ---
target_name = "Minelauva"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (193.90086927, 3.3974689)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (193.90086927, 3.3974689)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (193.90086927, 3.3974689)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (193.95508712, 3.37570977)>
Using SkyOffsetFrame for Star Minelauva 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 12.9267, 3.3975
Original RA(Deg)/Dec: 193.9009, 3.3975
New  RA(Hr)/Dec:  12.9303, 3.3757
New  RA(Deg)/Dec:  193.9551, 3.3757
Delta RA/DEC(min): -3.2531,       1.3055


In [30]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [31]:
star_magnitude = 3.4;  target_alt_name = "HD 112300" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M3'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 3.4: 12.08 s


In [32]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                       Name1*     Name2*   RA2000*   D2000*  Pmag~    Exp~  \
0      zosma_HD 102647_Typ_A3  HD 102647  168.5849  20.5020   2.14    3.98   
1   Arcturus_HD 124897_Typ_K1  HD 124897  213.9726  19.1606  -0.05    0.58   
2           Neptune_HD_Typ_NA         HD    0.1835  -0.6563   7.74  556.18   
3       Zosma_HD 97603_Typ_A4   HD 97603  168.5849  20.5020   2.56    5.76   
4  Minelauva_HD 112300_Typ_M3  HD 112300  193.9551   3.3757    3.4   12.08   

  Note1 Note2 NExp~ GetRef Temp  
0    NA    NA     1      0       
1    NA    NA     1      0       
2    NA    NA     1      0       
3    NA    NA     1      0       
4    NA    NA     1      0       


In [33]:
# --- Change User Inputs ---
target_name = "HD 138629"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (232.94576269, 40.89933486)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (232.94576269, 40.89933486)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (232.94576269, 40.89933486)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (233.01734459, 40.87755511)>
Using SkyOffsetFrame for Star HD 138629 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 15.5297, 40.8993
Original RA(Deg)/Dec: 232.9458, 40.8993
New  RA(Hr)/Dec:  15.5345, 40.8776
New  RA(Deg)/Dec:  233.0173, 40.8776
Delta RA/DEC(min): -4.2949,       1.3068


In [34]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [35]:
star_magnitude = 5.02;  target_alt_name = "Nu2 Boo" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A5'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 5.02: 50.46 s


In [36]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                       Name1*     Name2*   RA2000*   D2000*  Pmag~    Exp~  \
0      zosma_HD 102647_Typ_A3  HD 102647  168.5849  20.5020   2.14    3.98   
1   Arcturus_HD 124897_Typ_K1  HD 124897  213.9726  19.1606  -0.05    0.58   
2           Neptune_HD_Typ_NA         HD    0.1835  -0.6563   7.74  556.18   
3       Zosma_HD 97603_Typ_A4   HD 97603  168.5849  20.5020   2.56    5.76   
4  Minelauva_HD 112300_Typ_M3  HD 112300  193.9551   3.3757    3.4   12.08   
5    HD 138629_Nu2 Boo_Typ_A5    Nu2 Boo  233.0173  40.8776   5.02   50.46   

  Note1 Note2 NExp~ GetRef Temp  
0    NA    NA     1      0       
1    NA    NA     1      0       
2    NA    NA     1      0       
3    NA    NA     1      0       
4    NA    NA     1      0       
5    NA    NA     1      0       


In [37]:
# --- Change User Inputs ---
target_name = "HD 142105"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (236.01466071, 77.79449312)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (236.01466071, 77.79449312)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (236.01466071, 77.79449312)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (236.27021321, 77.77261753)>
Using SkyOffsetFrame for Star HD 142105 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 15.7343, 77.7945
Original RA(Deg)/Dec: 236.0147, 77.7945
New  RA(Hr)/Dec:  15.7513, 77.7726
New  RA(Deg)/Dec:  236.2702, 77.7726
Delta RA/DEC(min): -15.3332,       1.3125


In [38]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [39]:
star_magnitude = 4.29;  target_alt_name = "Zeta Umi" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A3'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 4.29: 26.50 s


In [40]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                       Name1*     Name2*   RA2000*   D2000*  Pmag~    Exp~  \
0      zosma_HD 102647_Typ_A3  HD 102647  168.5849  20.5020   2.14    3.98   
1   Arcturus_HD 124897_Typ_K1  HD 124897  213.9726  19.1606  -0.05    0.58   
2           Neptune_HD_Typ_NA         HD    0.1835  -0.6563   7.74  556.18   
3       Zosma_HD 97603_Typ_A4   HD 97603  168.5849  20.5020   2.56    5.76   
4  Minelauva_HD 112300_Typ_M3  HD 112300  193.9551   3.3757    3.4   12.08   
5    HD 138629_Nu2 Boo_Typ_A5    Nu2 Boo  233.0173  40.8776   5.02   50.46   
6   HD 142105_Zeta Umi_Typ_A3   Zeta Umi  236.2702  77.7726   4.29   26.50   

  Note1 Note2 NExp~ GetRef Temp  
0    NA    NA     1      0       
1    NA    NA     1      0       
2    NA    NA     1      0       
3    NA    NA     1      0       
4    NA    NA     1      0       
5    NA    NA     1      0       
6    NA    NA     1      0       


In [41]:
# --- Change User Inputs ---
target_name = "R Lyr"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (283.83375974, 43.94608958)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (283.83375974, 43.94608958)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (283.83375974, 43.94608958)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (283.90890485, 43.92430733)>
Using SkyOffsetFrame for Star R Lyr 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 18.9223, 43.9461
Original RA(Deg)/Dec: 283.8338, 43.9461
New  RA(Hr)/Dec:  18.9273, 43.9243
New  RA(Deg)/Dec:  283.9089, 43.9243
Delta RA/DEC(min): -4.5087,       1.3069


In [42]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [43]:
star_magnitude = 3.9;  target_alt_name = "HD 175865" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M5'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 3.9: 18.79 s


In [44]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                       Name1*     Name2*   RA2000*   D2000*  Pmag~    Exp~  \
0      zosma_HD 102647_Typ_A3  HD 102647  168.5849  20.5020   2.14    3.98   
1   Arcturus_HD 124897_Typ_K1  HD 124897  213.9726  19.1606  -0.05    0.58   
2           Neptune_HD_Typ_NA         HD    0.1835  -0.6563   7.74  556.18   
3       Zosma_HD 97603_Typ_A4   HD 97603  168.5849  20.5020   2.56    5.76   
4  Minelauva_HD 112300_Typ_M3  HD 112300  193.9551   3.3757    3.4   12.08   
5    HD 138629_Nu2 Boo_Typ_A5    Nu2 Boo  233.0173  40.8776   5.02   50.46   
6   HD 142105_Zeta Umi_Typ_A3   Zeta Umi  236.2702  77.7726   4.29   26.50   
7      R Lyr_HD 175865_Typ_M5  HD 175865  283.9089  43.9243    3.9   18.79   

  Note1 Note2 NExp~ GetRef Temp  
0    NA    NA     1      0       
1    NA    NA     1      0       
2    NA    NA     1      0       
3    NA    NA     1      0       
4    NA    NA     1      0       
5    NA    NA     1      0       
6    NA    NA     1      0       
7    NA    NA     1      

In [45]:
# --- Change User Inputs ---
target_name = "Zet1 Lyr"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (281.19315451, 37.60512165)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (281.19315451, 37.60512165)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (281.19315451, 37.60512165)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (281.26145235, 37.58334435)>
Using SkyOffsetFrame for Star Zet1 Lyr 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 18.7462, 37.6051
Original RA(Deg)/Dec: 281.1932, 37.6051
New  RA(Hr)/Dec:  18.7508, 37.5833
New  RA(Deg)/Dec:  281.2615, 37.5833
Delta RA/DEC(min): -4.0979,       1.3066


In [46]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [47]:
star_magnitude = 4.37;  target_alt_name = "HD 173648" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'NA'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 4.37: 28.44 s


In [48]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                       Name1*     Name2*   RA2000*   D2000*  Pmag~    Exp~  \
0      zosma_HD 102647_Typ_A3  HD 102647  168.5849  20.5020   2.14    3.98   
1   Arcturus_HD 124897_Typ_K1  HD 124897  213.9726  19.1606  -0.05    0.58   
2           Neptune_HD_Typ_NA         HD    0.1835  -0.6563   7.74  556.18   
3       Zosma_HD 97603_Typ_A4   HD 97603  168.5849  20.5020   2.56    5.76   
4  Minelauva_HD 112300_Typ_M3  HD 112300  193.9551   3.3757    3.4   12.08   
5    HD 138629_Nu2 Boo_Typ_A5    Nu2 Boo  233.0173  40.8776   5.02   50.46   
6   HD 142105_Zeta Umi_Typ_A3   Zeta Umi  236.2702  77.7726   4.29   26.50   
7      R Lyr_HD 175865_Typ_M5  HD 175865  283.9089  43.9243    3.9   18.79   
8   Zet1 Lyr_HD 173648_Typ_NA  HD 173648  281.2615  37.5833   4.37   28.44   

  Note1 Note2 NExp~ GetRef Temp  
0    NA    NA     1      0       
1    NA    NA     1      0       
2    NA    NA     1      0       
3    NA    NA     1      0       
4    NA    NA     1      0       
5    NA    NA  

In [49]:
# --- Change User Inputs ---
target_name = "P Cyg"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (304.44667489, 38.03293031)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (304.44667489, 38.03293031)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (304.44667489, 38.03293031)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (304.5153694, 38.0111527)>
Using SkyOffsetFrame for Star P Cyg 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 20.2964, 38.0329
Original RA(Deg)/Dec: 304.4467, 38.0329
New  RA(Hr)/Dec:  20.3010, 38.0112
New  RA(Deg)/Dec:  304.5154, 38.0112
Delta RA/DEC(min): -4.1217,       1.3067


In [50]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [51]:
star_magnitude = 4.82;  target_alt_name = "HD 193237" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B1'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 4.82: 42.30 s


In [52]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                       Name1*     Name2*   RA2000*   D2000*  Pmag~    Exp~  \
0      zosma_HD 102647_Typ_A3  HD 102647  168.5849  20.5020   2.14    3.98   
1   Arcturus_HD 124897_Typ_K1  HD 124897  213.9726  19.1606  -0.05    0.58   
2           Neptune_HD_Typ_NA         HD    0.1835  -0.6563   7.74  556.18   
3       Zosma_HD 97603_Typ_A4   HD 97603  168.5849  20.5020   2.56    5.76   
4  Minelauva_HD 112300_Typ_M3  HD 112300  193.9551   3.3757    3.4   12.08   
5    HD 138629_Nu2 Boo_Typ_A5    Nu2 Boo  233.0173  40.8776   5.02   50.46   
6   HD 142105_Zeta Umi_Typ_A3   Zeta Umi  236.2702  77.7726   4.29   26.50   
7      R Lyr_HD 175865_Typ_M5  HD 175865  283.9089  43.9243    3.9   18.79   
8   Zet1 Lyr_HD 173648_Typ_NA  HD 173648  281.2615  37.5833   4.37   28.44   
9      P Cyg_HD 193237_Typ_B1  HD 193237  304.5154  38.0112   4.82   42.30   

  Note1 Note2 NExp~ GetRef Temp  
0    NA    NA     1      0       
1    NA    NA     1      0       
2    NA    NA     1      0       
3    

In [53]:
# --- Change User Inputs ---
target_name = "Navi"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (22.18838359, -1.87361633)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (22.18838359, -1.87361633)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (22.18838359, -1.87361633)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (22.24253699, -1.89537311)>
Using SkyOffsetFrame for Star Navi 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 1.4792, -1.8736
Original RA(Deg)/Dec: 22.1884, -1.8736
New  RA(Hr)/Dec:  1.4828, -1.8954
New  RA(Deg)/Dec:  22.2425, -1.8954
Delta RA/DEC(min): -3.2492,       1.3054


In [54]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [55]:
star_magnitude = 2.47;  target_alt_name = "HD 5394" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B0'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 2.47: 5.32 s


In [56]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                        Name1*     Name2*   RA2000*   D2000*  Pmag~    Exp~  \
0       zosma_HD 102647_Typ_A3  HD 102647  168.5849  20.5020   2.14    3.98   
1    Arcturus_HD 124897_Typ_K1  HD 124897  213.9726  19.1606  -0.05    0.58   
2            Neptune_HD_Typ_NA         HD    0.1835  -0.6563   7.74  556.18   
3        Zosma_HD 97603_Typ_A4   HD 97603  168.5849  20.5020   2.56    5.76   
4   Minelauva_HD 112300_Typ_M3  HD 112300  193.9551   3.3757    3.4   12.08   
5     HD 138629_Nu2 Boo_Typ_A5    Nu2 Boo  233.0173  40.8776   5.02   50.46   
6    HD 142105_Zeta Umi_Typ_A3   Zeta Umi  236.2702  77.7726   4.29   26.50   
7       R Lyr_HD 175865_Typ_M5  HD 175865  283.9089  43.9243    3.9   18.79   
8    Zet1 Lyr_HD 173648_Typ_NA  HD 173648  281.2615  37.5833   4.37   28.44   
9       P Cyg_HD 193237_Typ_B1  HD 193237  304.5154  38.0112   4.82   42.30   
10         Navi_HD 5394_Typ_B0    HD 5394   22.2425  -1.8954   2.47    5.32   

   Note1 Note2 NExp~ GetRef Temp  
0     NA    NA  

In [57]:
# --- Change User Inputs ---
target_name = "Albireo"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (292.68031501, 27.95967363)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (292.68031501, 27.95967363)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (292.68031501, 27.95967363)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (292.74157872, 27.93790244)>
Using SkyOffsetFrame for Star Albireo 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 19.5120, 27.9597
Original RA(Deg)/Dec: 292.6803, 27.9597
New  RA(Hr)/Dec:  19.5161, 27.9379
New  RA(Deg)/Dec:  292.7416, 27.9379
Delta RA/DEC(min): -3.6758,       1.3063


In [58]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [59]:
star_magnitude = 3.21;  target_alt_name = "HD 183912" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K2'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 3.21: 10.22 s


In [60]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                        Name1*     Name2*   RA2000*   D2000*  Pmag~    Exp~  \
0       zosma_HD 102647_Typ_A3  HD 102647  168.5849  20.5020   2.14    3.98   
1    Arcturus_HD 124897_Typ_K1  HD 124897  213.9726  19.1606  -0.05    0.58   
2            Neptune_HD_Typ_NA         HD    0.1835  -0.6563   7.74  556.18   
3        Zosma_HD 97603_Typ_A4   HD 97603  168.5849  20.5020   2.56    5.76   
4   Minelauva_HD 112300_Typ_M3  HD 112300  193.9551   3.3757    3.4   12.08   
5     HD 138629_Nu2 Boo_Typ_A5    Nu2 Boo  233.0173  40.8776   5.02   50.46   
6    HD 142105_Zeta Umi_Typ_A3   Zeta Umi  236.2702  77.7726   4.29   26.50   
7       R Lyr_HD 175865_Typ_M5  HD 175865  283.9089  43.9243    3.9   18.79   
8    Zet1 Lyr_HD 173648_Typ_NA  HD 173648  281.2615  37.5833   4.37   28.44   
9       P Cyg_HD 193237_Typ_B1  HD 193237  304.5154  38.0112   4.82   42.30   
10         Navi_HD 5394_Typ_B0    HD 5394   22.2425  -1.8954   2.47    5.32   
11    Albireo_HD 183912_Typ_K2  HD 183912  292.7416 

In [61]:
star_magnitude = 5.11;  target_alt_name = "HD 183913" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B8'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 5.11: 54.63 s


In [62]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                        Name1*     Name2*   RA2000*   D2000*  Pmag~    Exp~  \
0       zosma_HD 102647_Typ_A3  HD 102647  168.5849  20.5020   2.14    3.98   
1    Arcturus_HD 124897_Typ_K1  HD 124897  213.9726  19.1606  -0.05    0.58   
2            Neptune_HD_Typ_NA         HD    0.1835  -0.6563   7.74  556.18   
3        Zosma_HD 97603_Typ_A4   HD 97603  168.5849  20.5020   2.56    5.76   
4   Minelauva_HD 112300_Typ_M3  HD 112300  193.9551   3.3757    3.4   12.08   
5     HD 138629_Nu2 Boo_Typ_A5    Nu2 Boo  233.0173  40.8776   5.02   50.46   
6    HD 142105_Zeta Umi_Typ_A3   Zeta Umi  236.2702  77.7726   4.29   26.50   
7       R Lyr_HD 175865_Typ_M5  HD 175865  283.9089  43.9243    3.9   18.79   
8    Zet1 Lyr_HD 173648_Typ_NA  HD 173648  281.2615  37.5833   4.37   28.44   
9       P Cyg_HD 193237_Typ_B1  HD 193237  304.5154  38.0112   4.82   42.30   
10         Navi_HD 5394_Typ_B0    HD 5394   22.2425  -1.8954   2.47    5.32   
11    Albireo_HD 183912_Typ_K2  HD 183912  292.7416 

In [63]:
# --- Change User Inputs ---
target_name = "Alioth"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (193.50728997, 55.95982296)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (193.50728997, 55.95982296)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (193.50728997, 55.95982296)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (193.6039242, 55.93802752)>
Using SkyOffsetFrame for Star Alioth 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 12.9005, 55.9598
Original RA(Deg)/Dec: 193.5073, 55.9598
New  RA(Hr)/Dec:  12.9069, 55.9380
New  RA(Deg)/Dec:  193.6039, 55.9380
Delta RA/DEC(min): -5.7981,       1.3077


In [64]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [65]:
star_magnitude = 1.77;  target_alt_name = "HD 112185" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A1'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 1.77: 2.87 s


In [66]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                        Name1*     Name2*   RA2000*   D2000*  Pmag~    Exp~  \
0       zosma_HD 102647_Typ_A3  HD 102647  168.5849  20.5020   2.14    3.98   
1    Arcturus_HD 124897_Typ_K1  HD 124897  213.9726  19.1606  -0.05    0.58   
2            Neptune_HD_Typ_NA         HD    0.1835  -0.6563   7.74  556.18   
3        Zosma_HD 97603_Typ_A4   HD 97603  168.5849  20.5020   2.56    5.76   
4   Minelauva_HD 112300_Typ_M3  HD 112300  193.9551   3.3757    3.4   12.08   
5     HD 138629_Nu2 Boo_Typ_A5    Nu2 Boo  233.0173  40.8776   5.02   50.46   
6    HD 142105_Zeta Umi_Typ_A3   Zeta Umi  236.2702  77.7726   4.29   26.50   
7       R Lyr_HD 175865_Typ_M5  HD 175865  283.9089  43.9243    3.9   18.79   
8    Zet1 Lyr_HD 173648_Typ_NA  HD 173648  281.2615  37.5833   4.37   28.44   
9       P Cyg_HD 193237_Typ_B1  HD 193237  304.5154  38.0112   4.82   42.30   
10         Navi_HD 5394_Typ_B0    HD 5394   22.2425  -1.8954   2.47    5.32   
11    Albireo_HD 183912_Typ_K2  HD 183912  292.7416 

In [67]:
# --- Change User Inputs ---
target_name = "Vega"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (279.23473479, 38.78368896)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (279.23473479, 38.78368896)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (279.23473479, 38.78368896)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (279.30414611, 38.7619108)>
Using SkyOffsetFrame for Star Vega 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 18.6156, 38.7837
Original RA(Deg)/Dec: 279.2347, 38.7837
New  RA(Hr)/Dec:  18.6203, 38.7619
New  RA(Deg)/Dec:  279.3041, 38.7619
Delta RA/DEC(min): -4.1647,       1.3067


In [68]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [69]:
star_magnitude = 0.02;  target_alt_name = "HD 172167" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A0'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 0.02: 0.61 s


In [70]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                        Name1*     Name2*   RA2000*   D2000*  Pmag~    Exp~  \
0       zosma_HD 102647_Typ_A3  HD 102647  168.5849  20.5020   2.14    3.98   
1    Arcturus_HD 124897_Typ_K1  HD 124897  213.9726  19.1606  -0.05    0.58   
2            Neptune_HD_Typ_NA         HD    0.1835  -0.6563   7.74  556.18   
3        Zosma_HD 97603_Typ_A4   HD 97603  168.5849  20.5020   2.56    5.76   
4   Minelauva_HD 112300_Typ_M3  HD 112300  193.9551   3.3757    3.4   12.08   
5     HD 138629_Nu2 Boo_Typ_A5    Nu2 Boo  233.0173  40.8776   5.02   50.46   
6    HD 142105_Zeta Umi_Typ_A3   Zeta Umi  236.2702  77.7726   4.29   26.50   
7       R Lyr_HD 175865_Typ_M5  HD 175865  283.9089  43.9243    3.9   18.79   
8    Zet1 Lyr_HD 173648_Typ_NA  HD 173648  281.2615  37.5833   4.37   28.44   
9       P Cyg_HD 193237_Typ_B1  HD 193237  304.5154  38.0112   4.82   42.30   
10         Navi_HD 5394_Typ_B0    HD 5394   22.2425  -1.8954   2.47    5.32   
11    Albireo_HD 183912_Typ_K2  HD 183912  292.7416 

In [71]:
# --- Change User Inputs ---
target_name = "Scheat"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (345.94357274, 28.08278712)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (345.94357274, 28.08278712)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (345.94357274, 28.08278712)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (346.00490648, 28.06101587)>
Using SkyOffsetFrame for Star Scheat 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 23.0629, 28.0828
Original RA(Deg)/Dec: 345.9436, 28.0828
New  RA(Hr)/Dec:  23.0670, 28.0610
New  RA(Deg)/Dec:  346.0049, 28.0610
Delta RA/DEC(min): -3.6800,       1.3063


In [72]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [73]:
star_magnitude = 2.42;  target_alt_name = "HD 217906" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M2'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 2.42: 5.09 s


In [74]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1

In [75]:
df['Name1*'] = df['Name1*'].str.replace(' ', '_')
mdf = df.set_index("Name1*")
mdf.to_csv(f"{data_folder_name}/{output_csvfilename}")
print(mdf)

                               Name2*   RA2000*   D2000*  Pmag~    Exp~ Note1  \
Name1*                                                                          
zosma_HD_102647_Typ_A3      HD 102647  168.5849  20.5020   2.14    3.98    NA   
Arcturus_HD_124897_Typ_K1   HD 124897  213.9726  19.1606  -0.05    0.58    NA   
Neptune_HD_Typ_NA                  HD    0.1835  -0.6563   7.74  556.18    NA   
Zosma_HD_97603_Typ_A4        HD 97603  168.5849  20.5020   2.56    5.76    NA   
Minelauva_HD_112300_Typ_M3  HD 112300  193.9551   3.3757    3.4   12.08    NA   
HD_138629_Nu2_Boo_Typ_A5      Nu2 Boo  233.0173  40.8776   5.02   50.46    NA   
HD_142105_Zeta_Umi_Typ_A3    Zeta Umi  236.2702  77.7726   4.29   26.50    NA   
R_Lyr_HD_175865_Typ_M5      HD 175865  283.9089  43.9243    3.9   18.79    NA   
Zet1_Lyr_HD_173648_Typ_NA   HD 173648  281.2615  37.5833   4.37   28.44    NA   
P_Cyg_HD_193237_Typ_B1      HD 193237  304.5154  38.0112   4.82   42.30    NA   
Navi_HD_5394_Typ_B0         

In [76]:
# --- Change User Inputs ---
target_name = "RR Lyra"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.4f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.4f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (291.366304, 42.78435924)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (291.366304, 42.78435924)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (291.366304, 42.78435924)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (291.4400247, 42.76257797)>
Using SkyOffsetFrame for Star RR Lyra 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 19.4244, 42.7844
Original RA(Deg)/Dec: 291.3663, 42.7844
New  RA(Hr)/Dec:  19.4293, 42.7626
New  RA(Deg)/Dec:  291.4400, 42.7626
Delta RA/DEC(min): -4.4232,       1.3069


In [77]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

results = make_api_request(query)
if results:
    print("Data received:", results)


Error: 404


In [78]:
star_magnitude = 7.5;  target_alt_name = "HD 182989" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A7_F8'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 7.5: 450.04 s


In [79]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1

In [80]:
df['Name1*'] = df['Name1*'].str.replace(' ', '_')
mdf = df.set_index("Name1*")
mdf.to_csv(f"{data_folder_name}/{output_csvfilename}")
print(mdf)

                                Name2*   RA2000*   D2000*  Pmag~    Exp~  \
Name1*                                                                     
zosma_HD_102647_Typ_A3       HD 102647  168.5849  20.5020   2.14    3.98   
Arcturus_HD_124897_Typ_K1    HD 124897  213.9726  19.1606  -0.05    0.58   
Neptune_HD_Typ_NA                   HD    0.1835  -0.6563   7.74  556.18   
Zosma_HD_97603_Typ_A4         HD 97603  168.5849  20.5020   2.56    5.76   
Minelauva_HD_112300_Typ_M3   HD 112300  193.9551   3.3757    3.4   12.08   
HD_138629_Nu2_Boo_Typ_A5       Nu2 Boo  233.0173  40.8776   5.02   50.46   
HD_142105_Zeta_Umi_Typ_A3     Zeta Umi  236.2702  77.7726   4.29   26.50   
R_Lyr_HD_175865_Typ_M5       HD 175865  283.9089  43.9243    3.9   18.79   
Zet1_Lyr_HD_173648_Typ_NA    HD 173648  281.2615  37.5833   4.37   28.44   
P_Cyg_HD_193237_Typ_B1       HD 193237  304.5154  38.0112   4.82   42.30   
Navi_HD_5394_Typ_B0            HD 5394   22.2425  -1.8954   2.47    5.32   
Albireo_HD_1

In [81]:
#try:
#	from googlesearch import search
#except ImportError:
#	print("No module named 'google' found")
#
# to search
#query = f'{target_name} Wikipedia Astronomy'
#
#for j in search(query, tld="co.in", num=10, stop=10, pause=2):
#	print(j)